# Workshop: Gemma from Scratch
## Notebook 4: Rotary Positional Embeddings (RoPE)

**Estimated Time: 15 minutes**

Transformers are permutation-invariant, meaning they don't naturally know the order of tokens. **Rotary Positional Embeddings (RoPE)** encode position by rotating the Query and Key vectors in 2D space. The angle of rotation depends on the position in the sequence.

### Learning Objectives:
1. Understand the intuition: encoding position as rotation.
2. Implement the 2D rotation math.
3. Learn why RoPE allows for better relative position modeling.

In [ ]:
import torch
import matplotlib.pyplot as plt
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

head_dim = 64
seq_len = 100
theta_base = 10000.0

### 1. Computing Theta

RoPE splits the embedding into pairs of dimensions $(x_1, x_2, x_3, x_4, ...)$. Each pair is rotated by a different frequency $\theta_i$.

In [ ]:
# Compute frequencies
inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2).float() / head_dim))
t = torch.arange(seq_len)

# Compute angles for each position and frequency
# (seq_len) outer_product (head_dim/2) -> (seq_len, head_dim/2)
freqs = torch.outer(t, inv_freq)

# We use cos and sin to perform the rotation
cos = torch.cos(freqs)
sin = torch.sin(freqs)

plt.plot(freqs[:10, 0].detach().numpy(), label='Dim 0')
plt.plot(freqs[:10, 1].detach().numpy(), label='Dim 2')
plt.title("Rotation Angles over Sequence Position")
plt.xlabel("Sequence Position")
plt.ylabel("Angle (radians)")
plt.legend()
plt.show()

### 2. The Rotation Logic

For a 2D vector $(x_1, x_2)$, the rotation by angle $\theta$ is:
$$x'_1 = x_1 \cos \theta - x_2 \sin \theta$$
$$x'_2 = x_1 \sin \theta + x_2 \cos \theta$$

In implementation, we often split the vector into two halves: $x_{left}$ and $x_{right}$.

In [ ]:
def apply_rope(x, cos, sin):
    # x shape: (B, H, T, D)
    d = x.shape[-1]
    # Split x into two halves
    x_left = x[..., : d // 2]
    x_right = x[..., d // 2 :]
    
    # The rotation formula
    x_rotated = torch.cat([-x_right, x_left], dim=-1)
    
    # cos/sin need to be broadcasted to match x shape
    return (x * cos) + (x_rotated * sin)

print("RoPE logic defined.")

### 3. Why is this good?

When we compute the dot product between a Query at position $m$ and a Key at position $n$, the rotation ensures the score only depends on the **relative distance** $(m - n)$.

### Exercise:
Visualise how the dot product between two rotated vectors changes as you increase their distance.

**Hints:**
1. Create two random vectors $v_1$ and $v_2$ of shape `(1, 1, 1, head_dim)`.
2. Rotate $v_1$ at position 0.
3. Rotate $v_2$ at positions 0, 1, 2, ..., 20.
4. Compute the dot product between the rotated $v_1$ and each rotated $v_2$.
5. Plot the dot product vs distance.

In [ ]:
# Scaffolding
v1 = torch.randn(1, 1, 1, head_dim)
v2 = torch.randn(1, 1, 1, head_dim)

distances = range(20)
dot_products = []

for d in distances:
    # Rotate v1 at pos 0
    # cos_0, sin_0 = cos[0:1], sin[0:1]
    # v1_rot = apply_rope(v1, cos_0, sin_0)
    
    # Rotate v2 at pos d
    # cos_d, sin_d = cos[d:d+1], sin[d:d+1]
    # v2_rot = apply_rope(v2, cos_d, sin_d)
    
    # Compute dot product
    # dot = ...
    dot_products.append(0.0) # Replace with actual dot product

plt.plot(distances, dot_products)
plt.xlabel("Distance (m - n)")
plt.ylabel("Dot Product")
plt.title("RoPE: Decay of Dot Product over Distance")
plt.show()

<details>
<summary><b>Click to see solution</b></summary>

```python
v1 = torch.randn(1, 1, 1, head_dim)
v2 = torch.randn(1, 1, 1, head_dim)
distances = range(20)
dot_products = []

for d in distances:
    v1_rot = apply_rope(v1, cos[0:1], sin[0:1])
    v2_rot = apply_rope(v2, cos[d:d+1], sin[d:d+1])
    dot = (v1_rot * v2_rot).sum().item()
    dot_products.append(dot)

plt.plot(distances, dot_products)
```
</details>